# Data Cleaning and Preparation

This notebook focuses on cleaning, standardizing and preparing the
nine Olist e-commerce datasets for further analysis.

The workflow includes:
1. Loading raw datasets
2. Data type standardization
3. Missing value analysis
4. Duplicate investigation
5. Data cleaning and transformation
6. Data validation
7. Exporting processed datasets

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [2]:
CURRENT_DIR = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name.lower() == "notebooks"
    else CURRENT_DIR
)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DATA_DIR)
print("Processed data directory:", PROCESSED_DATA_DIR)

print("Raw directory exists:", RAW_DATA_DIR.exists())

Project root: c:\Users\RAHUL\OneDrive\Documents\Desktop\Projects\EDA_projects\ecommerce-customer-intelligence
Raw data directory: c:\Users\RAHUL\OneDrive\Documents\Desktop\Projects\EDA_projects\ecommerce-customer-intelligence\data\raw
Processed data directory: c:\Users\RAHUL\OneDrive\Documents\Desktop\Projects\EDA_projects\ecommerce-customer-intelligence\data\processed
Raw directory exists: True


In [3]:
expected_files = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"
]

dataframes = {}

for filename in expected_files:
    file_path = RAW_DATA_DIR / filename

    if not file_path.exists():
        raise FileNotFoundError(f"Missing dataset: {filename}")

    dataframes[filename.replace(".csv", "")] = pd.read_csv(
        file_path,
        low_memory=False
    )

print("Total datasets loaded:", len(dataframes))

Total datasets loaded: 9


In [4]:
dataset_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Memory_MB": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2), 2
        )
    }
    for name, df in dataframes.items()
])

display(dataset_summary)

,Dataset,Rows,Columns,Memory_MB
0,olist_customers_dataset,99441,5,26.59
1,olist_geolocation_dataset,1000163,5,129.38
2,olist_orders_dataset,99441,8,52.94
3,olist_order_items_dataset,112650,7,35.99
4,olist_order_payments_dataset,103886,5,16.23
5,olist_order_reviews_dataset,99224,7,39.12
6,olist_products_dataset,32951,9,6.30
7,olist_sellers_dataset,3095,4,0.59
8,product_category_name_translation,71,2,0.01


## 2. Data Type and Missing Value Analysis

Inspect the data types and missing values across all nine datasets
to identify columns that require standardization or cleaning.

In [5]:

quality_summary = []

for name, df in dataframes.items():

    for column in df.columns:

        missing_count = df[column].isna().sum()

        quality_summary.append({
            "Dataset": name,
            "Column": column,
            "Data_Type": str(df[column].dtype),
            "Missing_Count": missing_count,
            "Missing_Percentage": round(
                missing_count / len(df) * 100, 2
            ),
            "Unique_Values": df[column].nunique()
        })

quality_summary = pd.DataFrame(quality_summary)

display(
    quality_summary.sort_values(
        ["Dataset", "Missing_Count"],
        ascending=[True, False]
    ).reset_index(drop=True)
)


,Dataset,Column,Data_Type,Missing_Count,Missing_Percentage,Unique_Values
0,olist_customers_dataset,customer_id,str,0,0.00,99441
1,olist_customers_dataset,customer_unique_id,str,0,0.00,96096
2,olist_customers_dataset,customer_zip_code_prefix,int64,0,0.00,14994
3,olist_customers_dataset,customer_city,str,0,0.00,4119
4,olist_customers_dataset,customer_state,str,0,0.00,27
5,olist_geolocation_dataset,geolocation_zip_code_prefix,int64,0,0.00,19015
6,olist_geolocation_dataset,geolocation_lat,float64,0,0.00,717372
7,olist_geolocation_dataset,geolocation_lng,float64,0,0.00,717615
8,olist_geolocation_dataset,geolocation_city,str,0,0.00,8011
9,olist_geolocation_dataset,geolocation_state,str,0,0.00,27


In [6]:

missing_summary = quality_summary[
    quality_summary["Missing_Count"] > 0
].copy()

display(
    missing_summary.sort_values(
        "Missing_Percentage",
        ascending=False
    ).reset_index(drop=True)
)


,Dataset,Column,Data_Type,Missing_Count,Missing_Percentage,Unique_Values
0,olist_order_reviews_dataset,review_comment_title,str,87656,88.34,4527
1,olist_order_reviews_dataset,review_comment_message,str,58247,58.70,36159
2,olist_orders_dataset,order_delivered_customer_date,str,2965,2.98,95664
3,olist_products_dataset,product_name_lenght,float64,610,1.85,66
4,olist_products_dataset,product_category_name,str,610,1.85,73
5,olist_products_dataset,product_description_lenght,float64,610,1.85,2960
6,olist_products_dataset,product_photos_qty,float64,610,1.85,19
7,olist_orders_dataset,order_delivered_carrier_date,str,1783,1.79,81018
8,olist_orders_dataset,order_approved_at,str,160,0.16,90733
9,olist_products_dataset,product_weight_g,float64,2,0.01,2204


## 3. Missing Value Investigation

Investigate whitespace-only text values and patterns of missing
product attributes before deciding how to handle missing data.

In [7]:

whitespace_summary = []

for name, df in dataframes.items():

    for column in df.select_dtypes(
        include=["object", "string"]
    ).columns:

        values = df[column].astype("string")

        whitespace_count = (
            values.notna() & values.str.strip().eq("")
        ).sum()

        if whitespace_count > 0:
            whitespace_summary.append({
                "Dataset": name,
                "Column": column,
                "Whitespace_Only_Count": whitespace_count
            })

whitespace_summary = pd.DataFrame(whitespace_summary)

display(whitespace_summary)


,Dataset,Column,Whitespace_Only_Count
0,olist_order_reviews_dataset,review_comment_title,2
1,olist_order_reviews_dataset,review_comment_message,27


In [8]:

products = dataframes["olist_products_dataset"]

product_columns = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

missing_pattern = (
    products[product_columns]
    .isna()
    .sum(axis=1)
    .value_counts()
    .sort_index()
    .rename_axis("Missing_Columns")
    .reset_index(name="Product_Count")
)

display(missing_pattern)

print(
    "Rows with all four primary descriptive attributes missing:",
    products[
        products[
            [
                "product_category_name",
                "product_name_lenght",
                "product_description_lenght",
                "product_photos_qty"
            ]
        ].isna().all(axis=1)
    ].shape[0]
)


,Missing_Columns,Product_Count
0,0,32340
1,4,610
2,8,1


Rows with all four primary descriptive attributes missing: 610


In [9]:

zip_columns = {
    "olist_customers_dataset": "customer_zip_code_prefix",
    "olist_geolocation_dataset": "geolocation_zip_code_prefix",
    "olist_sellers_dataset": "seller_zip_code_prefix"
}

for dataset_name, column in zip_columns.items():

    file_path = RAW_DATA_DIR / f"{dataset_name}.csv"

    dataframes[dataset_name] = pd.read_csv(
        file_path,
        dtype={column: "string"},
        low_memory=False
    )

    print(
        dataset_name,
        dataframes[dataset_name][column].dtype
    )


olist_customers_dataset string
olist_geolocation_dataset string
olist_sellers_dataset string


In [10]:

zip_summary = []

for dataset_name, column in zip_columns.items():

    values = dataframes[dataset_name][column].str.strip()

    zip_summary.append({
        "Dataset": dataset_name,
        "Minimum_Length": values.str.len().min(),
        "Maximum_Length": values.str.len().max(),
        "Five_Digit_Count": values.str.fullmatch(r"\d{5}").sum(),
        "Invalid_Count": (~values.str.fullmatch(r"\d{5}").fillna(False)).sum(),
        "Sample_Values": values.head(5).tolist()
    })

display(pd.DataFrame(zip_summary))


,Dataset,Minimum_Length,Maximum_Length,Five_Digit_Count,Invalid_Count,Sample_Values
0,olist_customers_dataset,5,5,99441,0,"[14409, 09790, 01151, 08775, 13056]"
1,olist_geolocation_dataset,5,5,1000163,0,"[01037, 01046, 01046, 01041, 01035]"
2,olist_sellers_dataset,5,5,3095,0,"[13023, 13844, 20031, 04195, 12914]"


In [11]:

products = dataframes["olist_products_dataset"]

descriptive_columns = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

physical_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

descriptive_missing = products[descriptive_columns].isna().all(axis=1)
physical_missing = products[physical_columns].isna().all(axis=1)

fully_missing = products[
    descriptive_missing & physical_missing
]

physical_only_missing = products[
    physical_missing & ~descriptive_missing
]

print("Products missing all eight attributes:", len(fully_missing))
print("Products missing only physical attributes:", len(physical_only_missing))

display(
    pd.concat([fully_missing, physical_only_missing])[
        ["product_id"] + descriptive_columns + physical_columns
    ]
)


Products missing all eight attributes: 1
Products missing only physical attributes: 1


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN


In [12]:
original_row_counts = {
    name: len(df)
    for name, df in dataframes.items()
}

print(pd.Series(original_row_counts, name="Original Rows"))

olist_customers_dataset                99441
olist_geolocation_dataset            1000163
olist_orders_dataset                   99441
olist_order_items_dataset             112650
olist_order_payments_dataset          103886
olist_order_reviews_dataset            99224
olist_products_dataset                 32951
olist_sellers_dataset                   3095
product_category_name_translation         71
Name: Original Rows, dtype: int64


In [13]:

text_cleaning_results = []

for name, df in dataframes.items():

    text_columns = df.select_dtypes(
        include=["object", "string"]
    ).columns

    for column in text_columns:

        cleaned = df[column].astype("string").str.strip()

        whitespace_count = (
            df[column].notna() & cleaned.eq("").fillna(False)
        ).sum()

        df[column] = cleaned.mask(cleaned.eq(""), pd.NA)

        if whitespace_count > 0:
            text_cleaning_results.append({
                "Dataset": name,
                "Column": column,
                "Whitespace_Values_Normalized": int(whitespace_count)
            })

display(pd.DataFrame(text_cleaning_results))


,Dataset,Column,Whitespace_Values_Normalized
0,olist_order_reviews_dataset,review_comment_title,2
1,olist_order_reviews_dataset,review_comment_message,27


In [14]:

datetime_columns = {
    "olist_orders_dataset": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],

    "olist_order_items_dataset": [
        "shipping_limit_date"
    ],

    "olist_order_reviews_dataset": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

date_validation_results = []
date_parse_issues = {}

for dataset_name, columns in datetime_columns.items():

    df = dataframes[dataset_name]

    for column in columns:

        original_values = df[column].copy()

        converted = pd.to_datetime(
            original_values,
            errors="coerce"
        )

        invalid_mask = (
            original_values.notna() & converted.isna()
        )

        invalid_count = int(invalid_mask.sum())

        date_validation_results.append({
            "Dataset": dataset_name,
            "Column": column,
            "Original_Missing": int(original_values.isna().sum()),
            "Successfully_Parsed": int(converted.notna().sum()),
            "Invalid_Values": invalid_count
        })

        if invalid_count > 0:
            date_parse_issues[(dataset_name, column)] = (
                original_values[invalid_mask].head(20).tolist()
            )

        df[column] = converted

display(pd.DataFrame(date_validation_results))

print("Date columns with parsing issues:", len(date_parse_issues))


,Dataset,Column,Original_Missing,Successfully_Parsed,Invalid_Values
0,olist_orders_dataset,order_purchase_timestamp,0,99441,0
1,olist_orders_dataset,order_approved_at,160,99281,0
2,olist_orders_dataset,order_delivered_carrier_date,1783,97658,0
3,olist_orders_dataset,order_delivered_customer_date,2965,96476,0
4,olist_orders_dataset,order_estimated_delivery_date,0,99441,0
5,olist_order_items_dataset,shipping_limit_date,0,112650,0
6,olist_order_reviews_dataset,review_creation_date,0,99224,0
7,olist_order_reviews_dataset,review_answer_timestamp,0,99224,0


Date columns with parsing issues: 0


In [15]:
reviews = dataframes["olist_order_reviews_dataset"]

display(
    reviews[
        ["review_comment_title", "review_comment_message"]
    ].isna().sum()
)

review_comment_title      87658
review_comment_message    58274
dtype: int64

In [16]:
cleaned_row_counts = {
    name: len(df)
    for name, df in dataframes.items()
}

row_comparison = pd.DataFrame({
    "Original_Rows": original_row_counts,
    "Current_Rows": cleaned_row_counts
})

row_comparison["Rows_Changed"] = (
    row_comparison["Original_Rows"]
    - row_comparison["Current_Rows"]
)

display(row_comparison)

,Original_Rows,Current_Rows,Rows_Changed
olist_customers_dataset,99441,99441,0
olist_geolocation_dataset,1000163,1000163,0
olist_orders_dataset,99441,99441,0
olist_order_items_dataset,112650,112650,0
olist_order_payments_dataset,103886,103886,0
olist_order_reviews_dataset,99224,99224,0
olist_products_dataset,32951,32951,0
olist_sellers_dataset,3095,3095,0
product_category_name_translation,71,71,0


In [17]:
products = dataframes["olist_products_dataset"]

product_numeric_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

numeric_validation = []

for column in product_numeric_columns:
    values = products[column].dropna()

    fractional_count = (values.mod(1) != 0).sum()

    numeric_validation.append({
        "Column": column,
        "Current_Dtype": str(products[column].dtype),
        "Missing_Values": products[column].isna().sum(),
        "Fractional_Values": fractional_count,
        "Can_Convert_To_Integer": fractional_count == 0
    })

display(pd.DataFrame(numeric_validation))

,Column,Current_Dtype,Missing_Values,Fractional_Values,Can_Convert_To_Integer
0,product_name_lenght,float64,610,0,True
1,product_description_lenght,float64,610,0,True
2,product_photos_qty,float64,610,0,True
3,product_weight_g,float64,2,0,True
4,product_length_cm,float64,2,0,True
5,product_height_cm,float64,2,0,True
6,product_width_cm,float64,2,0,True


In [18]:
for column in product_numeric_columns:
    products[column] = products[column].astype("Int64")

In [19]:
display(products[product_numeric_columns].dtypes)

display(
    products[product_numeric_columns].isna().sum()
)

product_name_lenght           Int64
product_description_lenght    Int64
product_photos_qty            Int64
product_weight_g              Int64
product_length_cm             Int64
product_height_cm             Int64
product_width_cm              Int64
dtype: object

product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [20]:
duplicate_summary = []

for dataset_name, df in dataframes.items():
    duplicate_count = df.duplicated().sum()

    duplicate_summary.append({
        "Dataset": dataset_name,
        "Total_Rows": len(df),
        "Exact_Duplicate_Rows": duplicate_count,
        "Duplicate_Percentage": round(
            duplicate_count / len(df) * 100, 2
        )
    })

duplicate_summary = pd.DataFrame(duplicate_summary)

display(
    duplicate_summary.sort_values(
        by="Exact_Duplicate_Rows",
        ascending=False
    )
)

,Dataset,Total_Rows,Exact_Duplicate_Rows,Duplicate_Percentage
1,olist_geolocation_dataset,1000163,261831,26.18
0,olist_customers_dataset,99441,0,0.00
2,olist_orders_dataset,99441,0,0.00
3,olist_order_items_dataset,112650,0,0.00
4,olist_order_payments_dataset,103886,0,0.00
5,olist_order_reviews_dataset,99224,0,0.00
6,olist_products_dataset,32951,0,0.00
7,olist_sellers_dataset,3095,0,0.00
8,product_category_name_translation,71,0,0.00


In [21]:
key_definitions = {
    "olist_customers_dataset": ["customer_id"],
    "olist_orders_dataset": ["order_id"],
    "olist_order_items_dataset": ["order_id", "order_item_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
    "olist_order_reviews_dataset": ["review_id", "order_id"],
    "olist_products_dataset": ["product_id"],
    "olist_sellers_dataset": ["seller_id"],
    "product_category_name_translation": ["product_category_name"]
}

key_validation_results = []

for dataset_name, key_columns in key_definitions.items():
    df = dataframes[dataset_name]

    missing_key_values = df[key_columns].isna().sum().sum()
    duplicate_key_rows = df.duplicated(
        subset=key_columns
    ).sum()

    key_validation_results.append({
        "Dataset": dataset_name,
        "Key_Columns": ", ".join(key_columns),
        "Missing_Key_Values": missing_key_values,
        "Duplicate_Key_Rows": duplicate_key_rows,
        "Key_Validation": (
            "PASS"
            if missing_key_values == 0 and duplicate_key_rows == 0
            else "REVIEW"
        )
    })

key_validation = pd.DataFrame(key_validation_results)

display(key_validation)

,Dataset,Key_Columns,Missing_Key_Values,Duplicate_Key_Rows,Key_Validation
0,olist_customers_dataset,customer_id,0,0,PASS
1,olist_orders_dataset,order_id,0,0,PASS
2,olist_order_items_dataset,"order_id, order_item_id",0,0,PASS
3,olist_order_payments_dataset,"order_id, payment_sequential",0,0,PASS
4,olist_order_reviews_dataset,"review_id, order_id",0,0,PASS
5,olist_products_dataset,product_id,0,0,PASS
6,olist_sellers_dataset,seller_id,0,0,PASS
7,product_category_name_translation,product_category_name,0,0,PASS


In [22]:
relationships = [
    ("olist_orders_dataset", "customer_id", "olist_customers_dataset", "customer_id"),
    ("olist_order_items_dataset", "order_id", "olist_orders_dataset", "order_id"),
    ("olist_order_items_dataset", "product_id", "olist_products_dataset", "product_id"),
    ("olist_order_items_dataset", "seller_id", "olist_sellers_dataset", "seller_id"),
    ("olist_order_payments_dataset", "order_id", "olist_orders_dataset", "order_id"),
    ("olist_order_reviews_dataset", "order_id", "olist_orders_dataset", "order_id")
]

relationship_results = []

for child_name, child_col, parent_name, parent_col in relationships:
    child_df = dataframes[child_name]
    parent_df = dataframes[parent_name]

    parent_values = set(parent_df[parent_col].dropna().unique())

    missing_count = child_df[child_col].isna().sum()

    unmatched_mask = (
        child_df[child_col].notna()
        & ~child_df[child_col].isin(parent_values)
    )

    unmatched_count = unmatched_mask.sum()
    unmatched_unique = child_df.loc[
        unmatched_mask, child_col
    ].nunique()

    relationship_results.append({
        "Child_Dataset": child_name,
        "Reference_Column": child_col,
        "Parent_Dataset": parent_name,
        "Missing_References": missing_count,
        "Unmatched_Rows": unmatched_count,
        "Unmatched_Unique_Values": unmatched_unique,
        "Validation": (
            "PASS"
            if missing_count == 0 and unmatched_count == 0
            else "REVIEW"
        )
    })

relationship_validation = pd.DataFrame(relationship_results)

display(relationship_validation)

,Child_Dataset,Reference_Column,Parent_Dataset,Missing_References,Unmatched_Rows,Unmatched_Unique_Values,Validation
0,olist_orders_dataset,customer_id,olist_customers_dataset,0,0,0,PASS
1,olist_order_items_dataset,order_id,olist_orders_dataset,0,0,0,PASS
2,olist_order_items_dataset,product_id,olist_products_dataset,0,0,0,PASS
3,olist_order_items_dataset,seller_id,olist_sellers_dataset,0,0,0,PASS
4,olist_order_payments_dataset,order_id,olist_orders_dataset,0,0,0,PASS
5,olist_order_reviews_dataset,order_id,olist_orders_dataset,0,0,0,PASS


In [23]:
lookup_relationships = [
    (
        "olist_customers_dataset",
        "customer_zip_code_prefix",
        "olist_geolocation_dataset",
        "geolocation_zip_code_prefix"
    ),
    (
        "olist_sellers_dataset",
        "seller_zip_code_prefix",
        "olist_geolocation_dataset",
        "geolocation_zip_code_prefix"
    ),
    (
        "olist_products_dataset",
        "product_category_name",
        "product_category_name_translation",
        "product_category_name"
    )
]

lookup_results = []

for child_name, child_col, parent_name, parent_col in lookup_relationships:
    child_df = dataframes[child_name]
    parent_df = dataframes[parent_name]

    parent_values = set(parent_df[parent_col].dropna().unique())

    missing_count = child_df[child_col].isna().sum()

    unmatched_mask = (
        child_df[child_col].notna()
        & ~child_df[child_col].isin(parent_values)
    )

    lookup_results.append({
        "Dataset": child_name,
        "Lookup_Column": child_col,
        "Missing_Values": missing_count,
        "Unmatched_Rows": unmatched_mask.sum(),
        "Unmatched_Unique_Values": child_df.loc[
            unmatched_mask, child_col
        ].nunique()
    })

lookup_validation = pd.DataFrame(lookup_results)

display(lookup_validation)

,Dataset,Lookup_Column,Missing_Values,Unmatched_Rows,Unmatched_Unique_Values
0,olist_customers_dataset,customer_zip_code_prefix,0,278,157
1,olist_sellers_dataset,seller_zip_code_prefix,0,7,7
2,olist_products_dataset,product_category_name,610,13,2


In [24]:
customers = dataframes["olist_customers_dataset"]
sellers = dataframes["olist_sellers_dataset"]
geolocation = dataframes["olist_geolocation_dataset"]
products = dataframes["olist_products_dataset"]
translations = dataframes["product_category_name_translation"]

geo_zip_codes = set(
    geolocation["geolocation_zip_code_prefix"].dropna().unique()
)

translated_categories = set(
    translations["product_category_name"].dropna().unique()
)

customer_unmatched = customers.loc[
    ~customers["customer_zip_code_prefix"].isin(geo_zip_codes),
    "customer_zip_code_prefix"
].value_counts()

seller_unmatched = sellers.loc[
    ~sellers["seller_zip_code_prefix"].isin(geo_zip_codes),
    "seller_zip_code_prefix"
].value_counts()

category_unmatched = products.loc[
    products["product_category_name"].notna()
    & ~products["product_category_name"].isin(translated_categories),
    "product_category_name"
].value_counts()

print("Unmatched customer ZIP prefixes:")
display(customer_unmatched)

print("Unmatched seller ZIP prefixes:")
display(seller_unmatched)

print("Unmatched product categories:")
display(category_unmatched)

Unmatched customer ZIP prefixes:


customer_zip_code_prefix
70686    15
72005    13
71919    10
73255     7
72300     6
         ..
12770     1
72338     1
68629     1
28530     1
55863     1
Name: count, Length: 157, dtype: Int64

Unmatched seller ZIP prefixes:


seller_zip_code_prefix
82040    1
91901    1
72580    1
02285    1
07412    1
71551    1
37708    1
Name: count, dtype: Int64

Unmatched product categories:


product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: Int64

In [25]:
numeric_rules = {
    "olist_order_items_dataset": {
        "order_item_id": (1, None),
        "price": (0, None),
        "freight_value": (0, None)
    },
    "olist_order_payments_dataset": {
        "payment_sequential": (1, None),
        "payment_installments": (1, None),
        "payment_value": (0, None)
    },
    "olist_order_reviews_dataset": {
        "review_score": (1, 5)
    },
    "olist_products_dataset": {
        "product_weight_g": (0, None),
        "product_length_cm": (0, None),
        "product_height_cm": (0, None),
        "product_width_cm": (0, None)
    },
    "olist_geolocation_dataset": {
        "geolocation_lat": (-90, 90),
        "geolocation_lng": (-180, 180)
    }
}

numeric_validation_results = []

for dataset_name, column_rules in numeric_rules.items():
    df = dataframes[dataset_name]

    for column, (lower, upper) in column_rules.items():
        values = df[column]

        out_of_range = pd.Series(False, index=df.index)

        if lower is not None:
            out_of_range |= values.lt(lower).fillna(False)

        if upper is not None:
            out_of_range |= values.gt(upper).fillna(False)

        invalid_count = (
            values.notna() & out_of_range
        ).sum()

        numeric_validation_results.append({
            "Dataset": dataset_name,
            "Column": column,
            "Minimum": values.min(),
            "Maximum": values.max(),
            "Missing_Values": values.isna().sum(),
            "Out_Of_Range_Values": invalid_count
        })

numeric_validation = pd.DataFrame(numeric_validation_results)

display(numeric_validation)

,Dataset,Column,Minimum,Maximum,Missing_Values,Out_Of_Range_Values
0,olist_order_items_dataset,order_item_id,1.000000,21.000000,0,0
1,olist_order_items_dataset,price,0.850000,6735.000000,0,0
2,olist_order_items_dataset,freight_value,0.000000,409.680000,0,0
3,olist_order_payments_dataset,payment_sequential,1.000000,29.000000,0,0
4,olist_order_payments_dataset,payment_installments,0.000000,24.000000,0,2
5,olist_order_payments_dataset,payment_value,0.000000,13664.080000,0,0
6,olist_order_reviews_dataset,review_score,1.000000,5.000000,0,0
7,olist_products_dataset,product_weight_g,0.000000,40425.000000,2,0
8,olist_products_dataset,product_length_cm,7.000000,105.000000,2,0
9,olist_products_dataset,product_height_cm,2.000000,105.000000,2,0


In [26]:
payments = dataframes["olist_order_payments_dataset"]

zero_installments = payments.loc[
    payments["payment_installments"] < 1
]

print("Records with invalid installment counts:")
display(
    zero_installments[
        [
            "order_id",
            "payment_sequential",
            "payment_type",
            "payment_installments",
            "payment_value"
        ]
    ]
)

print("Payment types involved:")
display(zero_installments["payment_type"].value_counts())

Records with invalid installment counts:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


Payment types involved:


payment_type
credit_card    2
Name: count, dtype: Int64

In [27]:
geolocation = dataframes["olist_geolocation_dataset"]

brazil_mask = (
    geolocation["geolocation_lat"].between(-35, 6)
    & geolocation["geolocation_lng"].between(-75, -32)
)

outside_brazil = geolocation.loc[~brazil_mask]

print("Total geolocation records:", len(geolocation))
print("Records outside approximate Brazil bounds:", len(outside_brazil))
print(
    "Unique ZIP prefixes affected:",
    outside_brazil["geolocation_zip_code_prefix"].nunique()
)

display(
    outside_brazil[
        [
            "geolocation_zip_code_prefix",
            "geolocation_lat",
            "geolocation_lng",
            "geolocation_city",
            "geolocation_state"
        ]
    ].head(20)
)

Total geolocation records: 1000163
Records outside approximate Brazil bounds: 29
Unique ZIP prefixes affected: 20


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG
695377,45936,38.323939,-6.775035,itabatan,BA


In [28]:
zero_order_ids = zero_installments["order_id"].unique()

print("All payment records for affected orders:")

display(
    payments[
        payments["order_id"].isin(zero_order_ids)
    ].sort_values(
        ["order_id", "payment_sequential"]
    )
)

orders = dataframes["olist_orders_dataset"]

print("Order details:")

display(
    orders.loc[
        orders["order_id"].isin(zero_order_ids),
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp"
        ]
    ]
)

All payment records for affected orders:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


Order details:


,order_id,order_status,order_purchase_timestamp
63782,744bade1fcf9ff3f31d860ace076d422,delivered,2018-04-22 11:34:42
66368,1a57108394169c0b47d8f876acc9ba2d,delivered,2018-05-15 16:25:14


In [29]:
outside_brazil = geolocation.loc[
    ~brazil_mask
].copy()

print("Total geolocation records:", len(geolocation))
print("Records outside approximate Brazil bounds:", len(outside_brazil))

print(
    "Percentage outside bounds:",
    round(len(outside_brazil) / len(geolocation) * 100, 2),
    "%"
)

print(
    "Unique affected ZIP prefixes:",
    outside_brazil["geolocation_zip_code_prefix"].nunique()
)

print("\nBreakdown by state:")

display(
    outside_brazil.groupby("geolocation_state").agg(
        Out_Of_Bounds_Rows=("geolocation_zip_code_prefix", "size"),
        Affected_ZIP_Prefixes=("geolocation_zip_code_prefix", "nunique"),
        Affected_Cities=("geolocation_city", "nunique")
    ).sort_values(
        "Out_Of_Bounds_Rows",
        ascending=False
    )
)

Total geolocation records: 1000163
Records outside approximate Brazil bounds: 29
Percentage outside bounds: 0.0 %
Unique affected ZIP prefixes: 20

Breakdown by state:


,Out_Of_Bounds_Rows,Affected_ZIP_Prefixes,Affected_Cities
geolocation_state,,,
PA,7,3,3
RJ,4,4,4
RS,4,2,2
BA,3,3,3
PR,3,2,2
MG,2,1,2
ES,2,1,2
AL,1,1,1
MT,1,1,1


In [30]:
geo_zip_analysis = geolocation.assign(
    outside_brazil=~brazil_mask
).groupby("geolocation_zip_code_prefix").agg(
    Total_Records=("outside_brazil", "size"),
    Invalid_Coordinates=("outside_brazil", "sum")
)

geo_zip_analysis["Valid_Coordinates"] = (
    geo_zip_analysis["Total_Records"]
    - geo_zip_analysis["Invalid_Coordinates"]
)

geo_zip_analysis["Invalid_Percentage"] = (
    geo_zip_analysis["Invalid_Coordinates"]
    / geo_zip_analysis["Total_Records"] * 100
).round(2)

display(
    geo_zip_analysis[
        geo_zip_analysis["Invalid_Coordinates"] > 0
    ].sort_values(
        "Invalid_Coordinates",
        ascending=False
    ).head(20)
)

,Total_Records,Invalid_Coordinates,Valid_Coordinates,Invalid_Percentage
geolocation_zip_code_prefix,,,,
68275,9,5,4,55.56
98780,139,3,136,2.16
35179,85,2,83,2.35
29654,4,2,2,50.00
83252,2,2,0,100.00
28165,4,1,3,25.00
28155,5,1,4,20.00
18243,1,1,0,100.00
28333,7,1,6,14.29


In [31]:
payments = dataframes["olist_order_payments_dataset"]
geolocation = dataframes["olist_geolocation_dataset"]

payments["has_installment_count_issue"] = (
    payments["payment_installments"] < 1
)

geolocation["is_outside_approx_brazil_bounds"] = (
    ~(
        geolocation["geolocation_lat"].between(-35, 6)
        & geolocation["geolocation_lng"].between(-75, -32)
    )
)

quality_flags_summary = pd.DataFrame([
    {
        "Dataset": "olist_order_payments_dataset",
        "Flag": "has_installment_count_issue",
        "Flagged_Rows": payments[
            "has_installment_count_issue"
        ].sum()
    },
    {
        "Dataset": "olist_geolocation_dataset",
        "Flag": "is_outside_approx_brazil_bounds",
        "Flagged_Rows": geolocation[
            "is_outside_approx_brazil_bounds"
        ].sum()
    }
])

display(quality_flags_summary)

print("Payment records flagged:")
display(
    payments.loc[
        payments["has_installment_count_issue"]
    ]
)

print("Geolocation records flagged:")
display(
    geolocation.loc[
        geolocation["is_outside_approx_brazil_bounds"],
        [
            "geolocation_zip_code_prefix",
            "geolocation_lat",
            "geolocation_lng",
            "geolocation_city",
            "geolocation_state"
        ]
    ]
)

,Dataset,Flag,Flagged_Rows
0,olist_order_payments_dataset,has_installment_count_issue,2
1,olist_geolocation_dataset,is_outside_approx_brazil_bounds,29


Payment records flagged:


,order_id,payment_sequential,payment_type,payment_installments,payment_value,has_installment_count_issue
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69,True
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94,True


Geolocation records flagged:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG
695377,45936,38.323939,-6.775035,itabatan,BA


In [32]:
final_integrity_results = []

for dataset_name, df in dataframes.items():
    final_integrity_results.append({
        "Dataset": dataset_name,
        "Original_Rows": original_row_counts[dataset_name],
        "Final_Rows": len(df),
        "Rows_Preserved": (
            original_row_counts[dataset_name] == len(df)
        ),
        "Columns": len(df.columns),
        "Total_Missing_Cells": int(df.isna().sum().sum())
    })

final_integrity = pd.DataFrame(final_integrity_results)

display(final_integrity)

print(
    "All row counts preserved:",
    final_integrity["Rows_Preserved"].all()
)

print(
    "Product nullable integer types correct:",
    all(
        str(dataframes["olist_products_dataset"][column].dtype) == "Int64"
        for column in product_numeric_columns
    )
)

print(
    "ZIP prefix string types correct:",
    all(
        str(dataframes[dataset][column].dtype) in ["string", "object"]
        for dataset, column in zip_columns.items()
    )
)

print(
    "Payment anomaly flags:",
    int(payments["has_installment_count_issue"].sum())
)

print(
    "Geolocation anomaly flags:",
    int(geolocation["is_outside_approx_brazil_bounds"].sum())
)

,Dataset,Original_Rows,Final_Rows,Rows_Preserved,Columns,Total_Missing_Cells
0,olist_customers_dataset,99441,99441,True,5,0
1,olist_geolocation_dataset,1000163,1000163,True,6,0
2,olist_orders_dataset,99441,99441,True,8,4908
3,olist_order_items_dataset,112650,112650,True,7,0
4,olist_order_payments_dataset,103886,103886,True,6,0
5,olist_order_reviews_dataset,99224,99224,True,7,145932
6,olist_products_dataset,32951,32951,True,9,2448
7,olist_sellers_dataset,3095,3095,True,4,0
8,product_category_name_translation,71,71,True,2,0


All row counts preserved: True
Product nullable integer types correct: True
ZIP prefix string types correct: True
Payment anomaly flags: 2
Geolocation anomaly flags: 29


In [33]:
export_results = []

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

for dataset_name, df in dataframes.items():

    output_path = PROCESSED_DATA_DIR / f"{dataset_name}.csv"

    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8"
    )

    export_results.append({
        "Dataset": dataset_name,
        "Rows_Exported": len(df),
        "Columns_Exported": len(df.columns),
        "File_Size_MB": round(
            output_path.stat().st_size / (1024 * 1024), 2
        ),
        "Export_Status": "SUCCESS" if output_path.exists() else "FAILED"
    })

export_summary = pd.DataFrame(export_results)

display(export_summary)

print("Total files exported:", len(export_summary))
print("All exports successful:", (export_summary["Export_Status"] == "SUCCESS").all())
print("Output directory:", PROCESSED_DATA_DIR)

,Dataset,Rows_Exported,Columns_Exported,File_Size_MB,Export_Status
0,olist_customers_dataset,99441,5,8.28,SUCCESS
1,olist_geolocation_dataset,1000163,6,63.20,SUCCESS
2,olist_orders_dataset,99441,8,15.84,SUCCESS
3,olist_order_items_dataset,112650,7,14.33,SUCCESS
4,olist_order_payments_dataset,103886,6,6.07,SUCCESS
5,olist_order_reviews_dataset,99224,7,13.39,SUCCESS
6,olist_products_dataset,32951,9,2.26,SUCCESS
7,olist_sellers_dataset,3095,4,0.16,SUCCESS
8,product_category_name_translation,71,2,0.00,SUCCESS


Total files exported: 9
All exports successful: True
Output directory: c:\Users\RAHUL\OneDrive\Documents\Desktop\Projects\EDA_projects\ecommerce-customer-intelligence\data\processed
